# Speaking score vs truth (Task 8)

Plots the lip-motion score against who is really speaking, to pick `speaking.threshold`.

Before running: `python eval/speaking_accuracy.py --extract` (slow, once) and then
`python eval/speaking_accuracy.py` (fast). The second command writes the
`output/speaking/<clip>_scores.csv` files this notebook reads; rerun it after changing
`config.yaml` or adding a label file, then rerun the notebook.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
SCORES = REPO / "output" / "speaking"
CLIP = "two_person_podcast"          # change me; see the list printed below

def load(clip):
    rows = list(csv.DictReader(open(SCORES / f"{clip}_scores.csv")))
    num = lambda r, k: float(r[k]) if r[k] != "" else np.nan
    return {k: np.array([num(r, k) for r in rows]) for k in ("t", "track_id", "ratio", "fluctuation", "prob", "pred", "truth")}

print("clips with scores:", sorted(p.name[:-11] for p in SCORES.glob("*_scores.csv")))
d = load(CLIP)
print(f"{CLIP}: {len(d['t'])} face-frames, tracks {sorted(set(d['track_id'].astype(int)))}, "
      f"truth known for {int(np.isfinite(d['truth']).sum())}")

## 1. Over time

Per track: the mouth-opening ratio, its fluctuation (what the score is built on) and the
resulting probability. Green bands are where the truth says "speaking"; the dashed line is the
threshold currently in `config.yaml`.

In [ ]:
import yaml
sp = yaml.safe_load(open(REPO / "config.yaml"))["speaking"]

tracks = [int(k) for k in sorted(set(d["track_id"].astype(int)))][:4]
fig, axes = plt.subplots(3 * len(tracks), 1, figsize=(13, 2.2 * 3 * len(tracks)), sharex=True, squeeze=False)
for i, tid in enumerate(tracks):
    m = d["track_id"] == tid
    t, truth = d["t"][m], d["truth"][m]
    for j, (key, label) in enumerate((("ratio", "mouth ratio"), ("fluctuation", "fluctuation"), ("prob", "probability"))):
        ax = axes[3 * i + j, 0]
        ax.plot(t, d[key][m], lw=0.9, color="tab:blue")
        if np.isfinite(truth).any():
            ax.fill_between(t, 0, 1, where=truth == 1, transform=ax.get_xaxis_transform(), color="tab:green", alpha=0.15, lw=0)
        if key == "fluctuation":
            ax.axhline(sp["threshold"], ls="--", color="tab:red", lw=0.8)
        if key == "prob":
            ax.axhline(sp["on_threshold"], ls=":", color="gray", lw=0.8); ax.axhline(sp["off_threshold"], ls=":", color="gray", lw=0.8)
            ax.fill_between(t, 0, 1, where=d["pred"][m] == 1, transform=ax.get_xaxis_transform(), color="tab:orange", alpha=0.12, lw=0)
        ax.set_ylabel(f"#{tid} {label}", fontsize=8)
axes[-1, 0].set_xlabel("time (s)")
fig.suptitle(f"{CLIP}: green = truth speaking, orange = predicted speaking", y=1.0)
plt.tight_layout(); plt.show()

## 2. How well does the fluctuation separate speaking from not speaking?

Two histograms should sit apart. Where they overlap is where a threshold has to trade missed
speech against false alarms.

In [ ]:
known = np.isfinite(d["truth"])
on, off = d["fluctuation"][known & (d["truth"] == 1)], d["fluctuation"][known & (d["truth"] == 0)]
bins = np.linspace(0, max(0.12, np.nanpercentile(d["fluctuation"], 99)), 60)
plt.figure(figsize=(9, 3.5))
plt.hist(off, bins, alpha=0.6, label=f"not speaking ({len(off)})", color="tab:gray", density=True)
plt.hist(on, bins, alpha=0.6, label=f"speaking ({len(on)})", color="tab:green", density=True)
plt.axvline(sp["threshold"], ls="--", color="tab:red", label=f"threshold {sp['threshold']}")
plt.xlabel("fluctuation of the mouth ratio over the window"); plt.ylabel("density"); plt.legend(); plt.title(CLIP); plt.show()

## 3. Threshold sweep

Accuracy, precision and recall of "fluctuation above threshold" per face-frame, without the
on/off hysteresis, so it is slightly pessimistic compared with `eval/speaking_accuracy.py`.
Pick the threshold near the top of the accuracy curve, put it in `config.yaml`
(`speaking.threshold`, and `slope` about a quarter of it), and rerun the evaluation.

In [ ]:
ths = np.linspace(0.004, 0.08, 77)
v, y = d["fluctuation"][known], d["truth"][known] == 1
acc = [((v > th) == y).mean() for th in ths]
prec = [(y & (v > th)).sum() / max((v > th).sum(), 1) for th in ths]
rec = [(y & (v > th)).sum() / max(y.sum(), 1) for th in ths]
plt.figure(figsize=(9, 3.5))
plt.plot(ths, acc, label="accuracy", lw=2); plt.plot(ths, prec, label="precision"); plt.plot(ths, rec, label="recall")
plt.axvline(sp["threshold"], ls="--", color="tab:red", label="config"); plt.axhline(0.85, ls=":", color="gray")
best = ths[int(np.argmax(acc))]
plt.xlabel("threshold on the fluctuation"); plt.legend(); plt.title(f"{CLIP}: best accuracy {max(acc):.1%} at {best:.3f}"); plt.show()
print(f"best threshold for {CLIP}: {best:.3f} -> accuracy {max(acc):.1%}; config has {sp['threshold']}")